In [1]:
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from IPython.display import HTML
import numpy as np
import requests
import re
import html
import codecs

In [2]:
from gmt_api_utils import fetch_measurement_data, get_run_ids, fetch_phase_stats, get_run_ids_by_names

In [3]:
REPOSITORY = 'https://github.com/envite-consulting/ollama-llm-energy-measurement'

In [4]:
# Add .api-key in root of repo with GMT-Api-Key
with open('../.api-key', 'r') as f:
    API_KEY = f.read().strip()

In [5]:
response = get_run_ids( 
    REPOSITORY, 
    API_KEY 
)

Request Status: 200 OK
Number of Runs: 39


In [6]:
selected_run_names = [
    'EMail Ollama Measurement Deepseek 1.5B v1.1.1',
    'EMail Ollama Measurement Gemma 3 4b v1.1',
    'EMail Ollama Measurement Gemma3 1b v1.1',
    'EMail Ollama Measurement Gemma 3 270M v1.1.2',
    'EMail Ollama Measurement Llama 3.2 3B v1.1',
    'EMail Ollama Measurement Llama 3.2 1B v1.1'
]

In [7]:
# Function to display the GMT data as a DataFrame
def display_gmt_data_as_dataframe(response):
    # Extract the 'data' property from the response
    data = response.json().get('data', [])

    # Create a DataFrame with the first 7 entries of each row
    df = pd.DataFrame(data, columns=['Run_ID', 'Name', 'Repo', 'Branch', 'Time', 'unknown', 'Usage_Scenario'] + [f'Extra_{i}' for i in range(len(data[0]) - 7)])
    df = df[['Run_ID', 'Name', 'Repo', 'Branch', 'Time', 'Usage_Scenario']]  # Keep only the required columns, excluding 'unknown'

    return df

available_runs = display_gmt_data_as_dataframe(response)

# Uncomment to see the dataframe
# available_runs  

In [8]:
run_ids_by_branch = get_run_ids_by_names(selected_run_names, available_runs)

run_ids_by_branch

{'EMail Ollama Measurement Deepseek 1.5B v1.1.1': ['98c5a741-dd13-4106-9853-70cf7e5a7b38',
  '71278ead-6a8f-47f5-a043-36abc7aa6a14',
  'da6a075f-f8e2-461a-b38b-1251db4f739e'],
 'EMail Ollama Measurement Gemma 3 4b v1.1': ['e8d0eba9-da7b-48ac-a326-5b822a02e3d6',
  'f9945116-628b-4f6d-813e-82ece5d330f1',
  'f0053916-336b-4965-b759-061228a9d6b1'],
 'EMail Ollama Measurement Gemma3 1b v1.1': ['980b81a4-59fb-4291-88db-b4966e2c843c',
  '0accabcd-4d81-4da4-bc56-45738f798a93',
  '04b7baab-232b-4dde-975f-573ef4f5cdc3'],
 'EMail Ollama Measurement Gemma 3 270M v1.1.2': ['cbadb27a-41f2-4b7c-9fa5-0288b4343740'],
 'EMail Ollama Measurement Llama 3.2 3B v1.1': ['c84b977e-99a1-4f9c-854c-e8e1fb1656db',
  '66ae01ed-dd8f-4e9f-a559-7cbb47c24568',
  '5821970a-9579-4fec-87c9-6ebc04124eeb'],
 'EMail Ollama Measurement Llama 3.2 1B v1.1': ['f1d58054-a67c-4cb7-88d4-21252237d64d',
  '9db51ef0-6b18-4de6-b63e-edb478b74878',
  'fc5b2aef-ec61-45a9-8f85-fe3770aee5a0']}

In [9]:
def get_logs(run_id, api_key):
    """
    Fetch logs for a given run ID from the GMT API.
    Args:
        run_id (str): The run ID to fetch logs for.
        api_key (str): The API key for authentication.
    Returns:
        str or None: The logs property from the response, or None if not found.
    """
    url = f"https://api.green-coding.io/v2/run/{run_id}"
    headers = {
        "x-authentication": api_key
    }
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        return response.json().get('data').get('logs')
    else:
        print(f"Failed to fetch logs for Run ID {run_id}: {response.status_code} {response.reason}")
        return None

In [10]:
logs_test = get_logs('e8d0eba9-da7b-48ac-a326-5b822a02e3d6',API_KEY)

logs_test

'gcb-ai-model_[&#x27;docker&#x27;, &#x27;exec&#x27;, &#x27;gcb-ai-model&#x27;, &#x27;ollama&#x27;, &#x27;pull&#x27;, &#x27;gemma3:4b&#x27;]:\nstderr: \x1b[?2026h\x1b[?25l\x1b[1Gpulling manifest ⠙ \x1b[K\x1b[?25h\x1b[?2026l\x1b[?2026h\x1b[?25l\x1b[1Gpulling manifest ⠙ \x1b[K\x1b[?25h\x1b[?2026l\x1b[?2026h\x1b[?25l\x1b[1Gpulling manifest ⠸ \x1b[K\x1b[?25h\x1b[?2026l\x1b[?2026h\x1b[?25l\x1b[1Gpulling manifest ⠸ \x1b[K\x1b[?25h\x1b[?2026l\x1b[?2026h\x1b[?25l\x1b[1Gpulling manifest ⠴ \x1b[K\x1b[?25h\x1b[?2026l\x1b[?2026h\x1b[?25l\x1b[1Gpulling manifest ⠴ \x1b[K\x1b[?25h\x1b[?2026l\x1b[?2026h\x1b[?25l\x1b[1Gpulling manifest ⠧ \x1b[K\x1b[?25h\x1b[?2026l\x1b[?2026h\x1b[?25l\x1b[1Gpulling manifest ⠧ \x1b[K\x1b[?25h\x1b[?2026l\x1b[?2026h\x1b[?25l\x1b[1Gpulling manifest ⠇ \x1b[K\x1b[?25h\x1b[?2026l\x1b[?2026h\x1b[?25l\x1b[1Gpulling manifest ⠏ \x1b[K\x1b[?25h\x1b[?2026l\x1b[?2026h\x1b[?25l\x1b[1Gpulling manifest ⠋ \x1b[K\x1b[?25h\x1b[?2026l\x1b[?2026h\x1b[?25l\x1b[1Gpulling manifest ⠹ \x1b[K\x1b[?

In [11]:
def get_logs(run_id, api_key):
    url = f"https://api.green-coding.io/v2/run/{run_id}"
    headers = {
        "x-authentication": api_key
    }
    response = requests.get(url, headers=headers)
    print(f"Run ID: {run_id}, Status Code: {response.status_code}")
    if response.status_code == 200:
        return response.json().get('data').get('logs')
    else:
        print(f"Failed to fetch logs for Run ID {run_id}: {response.status_code} {response.reason}")
        return None

def store_logs(run_ids_by_branch, api_key):
    logs_by_branch = {}
    for branch_name, run_ids in run_ids_by_branch.items():
        logs_by_branch[branch_name] = {}
        for run_id in run_ids:
            logs = get_logs(run_id, api_key)
            logs_by_branch[branch_name][run_id] = logs
    return logs_by_branch

logs = store_logs(run_ids_by_branch, API_KEY)

Run ID: 98c5a741-dd13-4106-9853-70cf7e5a7b38, Status Code: 200
Run ID: 71278ead-6a8f-47f5-a043-36abc7aa6a14, Status Code: 200
Run ID: da6a075f-f8e2-461a-b38b-1251db4f739e, Status Code: 200
Run ID: e8d0eba9-da7b-48ac-a326-5b822a02e3d6, Status Code: 200
Run ID: f9945116-628b-4f6d-813e-82ece5d330f1, Status Code: 200
Run ID: f0053916-336b-4965-b759-061228a9d6b1, Status Code: 200


Run ID: 980b81a4-59fb-4291-88db-b4966e2c843c, Status Code: 200
Run ID: 0accabcd-4d81-4da4-bc56-45738f798a93, Status Code: 200
Run ID: 04b7baab-232b-4dde-975f-573ef4f5cdc3, Status Code: 200
Run ID: cbadb27a-41f2-4b7c-9fa5-0288b4343740, Status Code: 200
Run ID: c84b977e-99a1-4f9c-854c-e8e1fb1656db, Status Code: 200
Run ID: 66ae01ed-dd8f-4e9f-a559-7cbb47c24568, Status Code: 200
Run ID: 5821970a-9579-4fec-87c9-6ebc04124eeb, Status Code: 200
Run ID: f1d58054-a67c-4cb7-88d4-21252237d64d, Status Code: 200
Run ID: 9db51ef0-6b18-4de6-b63e-edb478b74878, Status Code: 200
Run ID: fc5b2aef-ec61-45a9-8f85-fe3770aee5a0, Status Code: 200


> ℹ️ set `seperator_string` as the beginning of the prompt to properly slice the logs

In [12]:
seperator_string = 'Write an email to your manager explaining that you will work remotely tomorrow due to a family emergency'

In [ ]:
def isolate_phases_in_logs(logs, seperator_string):
    """
    Splits the logs into phases using the seperator_string.
    Returns a list of substrings, each starting with seperator_string and ending before the next occurrence.
    """
    phases = []
    start = 0
    while True:
        start = logs.find(seperator_string, start)
        if start == -1:
            break
        end = logs.find(seperator_string, start + len(seperator_string))
        if end == -1:
            phases.append(logs[start:])
            break
        else:
            phases.append(logs[start:end])
            start = end
    return phases

In [16]:
def isolate_phases_for_all_runs(logs, seperator_string):
    """
    Applies isolate_phases_in_logs to each run in logs.
    Returns a nested dictionary with the same structure as logs,
    where each run_id maps to a list of isolated phases.
    """
    phases_by_branch = {}
    for branch_name, runs in logs.items():
        phases_by_branch[branch_name] = {}
        for run_id, log_text in runs.items():
            if log_text is not None:
                phases = isolate_phases_in_logs(log_text, seperator_string)
            else:
                phases = []
            phases_by_branch[branch_name][run_id] = phases
    return phases_by_branch

filtered_logs = isolate_phases_for_all_runs(logs, seperator_string)

In [17]:
ANSI_RE = re.compile(r'\x1B\[[0-?]*[ -/]*[@-~]')

def to_human_readable(s: str) -> str:
    """
    Convert a log-like string containing HTML entities, backslash escapes,
    and ANSI terminal sequences into a clean, human-readable multi-line string.
    """
    # 1) Decode HTML entities (e.g., &#x27; → ', &quot; → ")
    s = html.unescape(s)

    # 2) Interpret Python-style escape sequences (\n, \t, \x1b, \uXXXX, etc.)
    #    Use unicode_escape decoding; if it fails, keep the original.
    try:
        s = codecs.decode(s, 'unicode_escape')
    except Exception:
        pass  # fall back to whatever we have

    # 3) Strip ANSI escape/control codes (cursor hide/show, colors, etc.)
    s = ANSI_RE.sub('', s)

    # 4) Remove other non-printable control chars except tab/newline
    s = re.sub(r'[\x00-\x08\x0b-\x0c\x0e-\x1f\x7f]', '', s)

    # 5) Normalize line endings
    s = s.replace('\r\n', '\n').replace('\r', '\n')

    # Optional: trim leading/trailing blank lines
    return s.strip()


In [18]:
def human_readable_logs(filtered_logs):
    """
    Applies to_human_readable to each phase in filtered_logs.
    Returns a nested dictionary with the same structure as filtered_logs,
    where each phase string is converted to human-readable format.
    """
    readable_by_branch = {}
    for branch_name, runs in filtered_logs.items():
        readable_by_branch[branch_name] = {}
        for run_id, phases in runs.items():
            readable_phases = [to_human_readable(phase) for phase in phases]
            readable_by_branch[branch_name][run_id] = readable_phases
    return readable_by_branch

readable_logs = human_readable_logs(filtered_logs)

In [19]:
import os
import json
from datetime import datetime

def save_logs_to_json(logs, filename_prefix, folder='logs'):
    """
    Saves the logs dictionary to a JSON file in the specified folder.
    The filename will be the current date and time in YYYY-MM-DD_HH-MM-SS format.
    """
    if not os.path.exists(folder):
        os.makedirs(folder)
    filename = datetime.now().strftime(f'{filename_prefix}-%Y-%m-%d_%H-%M-%S') + '.json'
    filepath = os.path.join(folder, filename)
    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(logs, f, ensure_ascii=False, indent=2)
    print(f"Logs saved to {filepath}")

save_logs_to_json(readable_logs,'test_prefix')

Logs saved to logs/test_prefix-2025-08-22_10-34-00.json
